In [ ]:
# Assignment 2: From Trees to Neural Networks
# COMS 4995 Applied Machine Learning — Spring 2026

import warnings
warnings.filterwarnings("ignore")

from src.utils import set_global_seed, setup_plotting, timer
from src.data_preparation import prepare_data, load_data, inspect_data, clean_data
from src.gbdt_model import train_gbdt, tune_gbdt, plot_training_validation_loss, plot_feature_importance, plot_learning_rate_comparison
from src.mlp_model import train_mlp, tune_mlp, plot_training_loss_curve, plot_depth_width_comparison
from src.mlp_model import plot_learning_rate_comparison as mlp_plot_lr
from src.evaluation import (
    build_comparison_table, plot_confusion_matrices, plot_pr_curves,
    print_classification_reports,
)

set_global_seed()
setup_plotting()
print("All modules loaded successfully.")

In [ ]:
# === 1. DATA PREPARATION ===
data = prepare_data()

X_train, X_val, X_test = data["X_train"], data["X_val"], data["X_test"]
y_train, y_val, y_test = data["y_train"], data["y_val"], data["y_test"]
X_train_scaled = data["X_train_scaled"]
X_val_scaled = data["X_val_scaled"]
X_test_scaled = data["X_test_scaled"]

print(f"\nFeature count: {len(data['feature_names'])}")
print(f"Class balance — Positive: {y_train.mean():.3f}, Negative: {1 - y_train.mean():.3f}")

In [ ]:
# === 2. GRADIENT BOOSTED DECISION TREES ===

import time

# Hyperparameter tuning
print("Tuning GBDT hyperparameters...")
start = time.time()
gbdt_model, gbdt_search = tune_gbdt(X_train, y_train, X_val, y_val)
gbdt_time = time.time() - start
print(f"\nGBDT total tuning + training time: {gbdt_time:.2f}s")

In [ ]:
# GBDT Visualizations
print("--- Training vs. Validation Loss ---")
plot_training_validation_loss(gbdt_model)

print("\n--- Feature Importance ---")
plot_feature_importance(gbdt_model)

print("\n--- Learning Rate Comparison ---")
plot_learning_rate_comparison(X_train, y_train, X_val, y_val)

In [ ]:
# === 3. MULTI-LAYER PERCEPTRON ===

print("Tuning MLP hyperparameters...")
start = time.time()
mlp_model, mlp_search = tune_mlp(X_train_scaled, y_train)
mlp_time = time.time() - start
print(f"\nMLP total tuning + training time: {mlp_time:.2f}s")

In [ ]:
# MLP Visualizations
print("--- Training Loss Curve ---")
plot_training_loss_curve(mlp_model)

print("\n--- Depth/Width Comparison ---")
plot_depth_width_comparison(X_train_scaled, y_train, X_val_scaled, y_val)

print("\n--- Learning Rate Comparison ---")
mlp_plot_lr(X_train_scaled, y_train, X_val_scaled, y_val)

In [ ]:
# === 4. GBDT vs MLP COMPARISON ===

# Predictions
gbdt_pred = gbdt_model.predict(X_test)
gbdt_prob = gbdt_model.predict_proba(X_test)[:, 1]

mlp_pred = mlp_model.predict(X_test_scaled)
mlp_prob = mlp_model.predict_proba(X_test_scaled)[:, 1]

# Comparison table
comparison = build_comparison_table(
    y_test, gbdt_pred, gbdt_prob, mlp_pred, mlp_prob, gbdt_time, mlp_time
)

In [ ]:
# === 5. EVALUATION & VISUALIZATION ===

print("--- Classification Reports ---")
print_classification_reports(y_test, gbdt_pred, mlp_pred)

print("\n--- Confusion Matrices ---")
plot_confusion_matrices(y_test, gbdt_pred, mlp_pred)

print("\n--- Precision-Recall Curves ---")
plot_pr_curves(y_test, gbdt_prob, mlp_prob)

In [ ]:
# === SUMMARY ===
print("\n" + "="*60)
print("ASSIGNMENT 2 COMPLETE — All visualizations saved to figures/")
print("="*60)
print(f"\nBest GBDT params: {gbdt_search.best_params_}")
print(f"Best MLP params:  {mlp_search.best_params_}")
print(f"\nGBDT Test F1: {comparison.loc['F1-score', 'GBDT (XGBoost)']}")
print(f"MLP  Test F1: {comparison.loc['F1-score', 'MLP (sklearn)']}")